# Preprocessing and Neural Inputs Trial

This notebook experiments with the package preprocessing stage and `neural_inputs.npy` packaging. Keep pipeline logic in `aquinas_toolkit`; use this notebook only to run and inspect current preprocess artifacts.

## Setup

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

repo_root = Path.cwd()
if not (repo_root / "aquinas_toolkit").exists():
    repo_root = next(path for path in Path.cwd().parents if (path / "aquinas_toolkit").exists())
sys.path.insert(0, str(repo_root))

from aquinas_toolkit.preprocessing import open_preprocess_store

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)
repo_root

## Active Preprocessing Configuration

In [ ]:
config_path = repo_root / "configs" / "default.yaml"
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
pre_cfg = config["preprocessing"]

pd.DataFrame(
    [
        {"variable": "sampling_rate_hz", "value": pre_cfg.get("sampling_rate_hz")},
        {"variable": "sensor_selection.decks", "value": pre_cfg.get("sensor_selection", {}).get("decks")},
        {"variable": "strain.locations", "value": pre_cfg.get("strain", {}).get("locations")},
        {"variable": "strain.peak_window_half_samples", "value": pre_cfg.get("strain", {}).get("peak_window_half_samples")},
        {"variable": "acc.min_aligned_samples", "value": pre_cfg.get("acc", {}).get("min_aligned_samples")},
        {"variable": "acc.fft_padding", "value": pre_cfg.get("acc", {}).get("fft_padding")},
        {"variable": "qc.flat_range_tolerance", "value": pre_cfg.get("qc", {}).get("flat_range_tolerance")},
        {"variable": "qc.mad_warning_threshold", "value": pre_cfg.get("qc", {}).get("mad_warning_threshold")},
        {"variable": "qc.mad_severe_threshold", "value": pre_cfg.get("qc", {}).get("mad_severe_threshold")},
    ]
)

## Run Preprocessing

In [ ]:
RUN_PIPELINE = False
RUN_NAME = "preprocessing-neural-trial"

if RUN_PIPELINE:
    command = [sys.executable, "-m", "aquinas_toolkit.cli", "run", "preprocess"]
    if RUN_NAME:
        command.extend(["--name", RUN_NAME])
    result = subprocess.run(command, cwd=repo_root, check=True, text=True, capture_output=True)
    print(result.stdout)
else:
    print("Set RUN_PIPELINE = True to execute preprocessing from this notebook.")

## Load Latest Preprocess Run

In [ ]:
RUN_ID = None
latest_path = repo_root / "results" / "latest.json"
if RUN_ID is None:
    latest = json.loads(latest_path.read_text(encoding="utf-8"))
    RUN_ID = latest["run_id"]

run_dir = repo_root / "results" / RUN_ID
preprocess_dir = run_dir / "stages" / "preprocess"
report_dir = preprocess_dir / "report"

paths = {
    "run_dir": run_dir,
    "preprocess_dir": preprocess_dir,
    "neural_inputs": preprocess_dir / "neural_inputs.npy",
    "neural_summary": report_dir / "neural_input_summary.json",
    "event_ids": report_dir / "event_ids.npy",
}
paths

## Neural Tensor and Sensor Numbering

In [ ]:
neural_inputs = np.load(preprocess_dir / "neural_inputs.npy")
sensor_map = pd.read_csv(report_dir / "sensor_map.csv")
input_slices = json.loads((report_dir / "input_slices.json").read_text(encoding="utf-8"))
sensor_ids = json.loads((report_dir / "sensor_ids.json").read_text(encoding="utf-8"))

print("neural_inputs shape:", neural_inputs.shape)
display(pd.DataFrame(input_slices).T)
display(sensor_map.loc[sensor_map["include_flag"].astype(bool)].sort_values("global_model_channel_index"))

## Coverage Summary

In [ ]:
neural_summary = json.loads((report_dir / "neural_input_summary.json").read_text(encoding="utf-8"))
frequency_bins = np.load(report_dir / "frequency_bins.npy")

coverage_overview = pd.Series(
    {
        "retained_preprocess_events_checked": neural_summary.get("total_retained_preprocess_events"),
        "complete_selected_sensor_coverage_events": neural_summary.get("complete_selected_sensor_coverage_events"),
        "events_excluded_incomplete_selected_sensor_coverage": neural_summary.get("events_excluded_incomplete_selected_sensor_coverage"),
        "events_excluded_packaging_constraints": neural_summary.get("events_excluded_packaging_constraints"),
        "packaged_neural_events": neural_summary.get("retained_events"),
        "input_vector_width": neural_summary.get("neural_inputs_shape", [0, 0])[1],
        "acc_frequency_bins": len(frequency_bins),
    },
    name="value",
)

display(coverage_overview.to_frame())

In [ ]:
plot_series = pd.Series(
    {
        "complete_selected_sensor_coverage": neural_summary.get("complete_selected_sensor_coverage_events", 0),
        "excluded_incomplete_coverage": neural_summary.get("events_excluded_incomplete_selected_sensor_coverage", 0),
        "excluded_packaging_constraints": neural_summary.get("events_excluded_packaging_constraints", 0),
        "packaged_neural_events": neural_summary.get("retained_events", 0),
    }
)

ax = plot_series.plot(kind="bar", figsize=(10, 4), title="Neural-input event counts")
ax.set_ylabel("events")
ax.tick_params(axis="x", labelrotation=25)
plt.tight_layout()

## Selected ACC-Z Sensors

In [ ]:
selected_acc_z = sensor_map.loc[
    sensor_map["include_flag"].astype(bool) & sensor_map["sensor_type"].eq("ACC_Z")
][
    ["sensor_name", "model_channel_id", "global_model_channel_index"]
]

display(selected_acc_z.sort_values("global_model_channel_index").reset_index(drop=True))

## Events Used for Neural Inputs

In [ ]:
event_ids = np.load(report_dir / "event_ids.npy", allow_pickle=False)

pd.DataFrame({"event_id": event_ids[:50]})

## Retained Sample Preview

In [ ]:
if len(neural_inputs):
    sample = neural_inputs[0]
    strain_slice = input_slices["strain"]
    acc_slice = input_slices["acc_z_frequency"]
    strain_block = sample[strain_slice["start"] : strain_slice["stop"]].reshape(strain_slice["shape"])
    acc_block = sample[acc_slice["start"] : acc_slice["stop"]].reshape(acc_slice["shape"])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(strain_block)
    axes[0].set_title("Strain peak windows")
    axes[0].set_xlabel("sample")
    axes[1].plot(acc_block)
    axes[1].set_title("ACC-Z frequency inputs")
    axes[1].set_xlabel("frequency bin index")
    fig.tight_layout()
else:
    print("No retained neural samples in this run.")